Cell 1: Imports & Directory Paths

In [2]:
import os
import tarfile
import glob
import scanpy as sc
import pandas as pd
import numpy as np
from scipy.io import mmread
import anndata as ad

DATA_DIR = "./data"
PROCESSED_DIR = "./processed_data"

os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Directories initialized successfully!")

C:\Users\omar_\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Directories initialized successfully!


Updated Cell 2: Check Extracted Data Directory

In [2]:
# Check whether files are directly in data or in raw_files subfolder
if len(glob.glob(os.path.join(DATA_DIR, "*matrix.mtx.gz"))) > 0:
    search_dir = DATA_DIR
    print(f"Found extracted .gz files directly in '{DATA_DIR}'!")
elif os.path.exists(os.path.join(DATA_DIR, "raw_files")):
    search_dir = os.path.join(DATA_DIR, "raw_files")
    print(f"Found extracted .gz files in '{search_dir}'!")
else:
    raise FileNotFoundError("Could not locate .gz sample matrices in data folder.")

Found extracted .gz files directly in './data'!


Cell 3: Load, Tag Metadata & Concatenate Samples

In [3]:
matrix_files = glob.glob(os.path.join(search_dir, "*matrix.mtx.gz"))
print(f"Found {len(matrix_files)} sample matrices.")

tissue_map = {
    "N": "Paratumor",
    "PT": "Primary Tumor",
    "LN": "LN Metastasis",
    "RAI": "RAI-Refractory Metastasis",
    "RAIR": "RAI-Refractory Metastasis"
}

adata_list = []

for mtx_file in sorted(matrix_files):
    base_prefix = mtx_file.replace("_matrix.mtx.gz", "")
    barcodes_file = f"{base_prefix}_barcodes.tsv.gz"
    features_file = f"{base_prefix}_features.tsv.gz"
    
    filename = os.path.basename(mtx_file)
    parts = filename.split("_")
    gsm_id = parts[0]
    sample_label = "_".join(parts[1:-1])
    patient_id = parts[1]
    tissue_code = parts[2] if len(parts) > 2 else "Unknown"
    tissue_type = tissue_map.get(tissue_code, tissue_code)
    
    print(f"Loading {sample_label} | Patient: {patient_id} | Stage: {tissue_type}")
    
    mtx = mmread(mtx_file).tocsr().T
    barcodes = pd.read_csv(barcodes_file, header=None, sep="\t")[0].values
    features = pd.read_csv(features_file, header=None, sep="\t")
    gene_names = features[1].values if features.shape[1] > 1 else features[0].values

    adata_sample = sc.AnnData(
        X=mtx,
        obs=pd.DataFrame(index=[f"{sample_label}_{b}" for b in barcodes]),
        var=pd.DataFrame(index=gene_names)
    )
    
    adata_sample.var_names_make_unique()
    adata_sample.obs["sample_id"] = sample_label
    adata_sample.obs["gsm_id"] = gsm_id
    adata_sample.obs["patient_id"] = patient_id
    adata_sample.obs["tissue_type"] = tissue_type
    
    adata_list.append(adata_sample)

print("\nConcatenating into master AnnData object...")
adata_master = ad.concat(adata_list, merge="same")
print("Concatenation complete!")

Found 23 sample matrices.
Loading PTC1_T | Patient: PTC1 | Stage: T


C:\Users\omar_\AppData\Local\Temp\ipykernel_15732\4137921281.py:34: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_sample = sc.AnnData(


Loading PTC1_P | Patient: PTC1 | Stage: P


C:\Users\omar_\AppData\Local\Temp\ipykernel_15732\4137921281.py:34: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_sample = sc.AnnData(


Loading PTC2_T | Patient: PTC2 | Stage: T


C:\Users\omar_\AppData\Local\Temp\ipykernel_15732\4137921281.py:34: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_sample = sc.AnnData(


... (37 outputs left out)

C:\Users\omar_\AppData\Local\Temp\ipykernel_15732\4137921281.py:34: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_sample = sc.AnnData(


Loading PTC11_SC | Patient: PTC11 | Stage: SC


C:\Users\omar_\AppData\Local\Temp\ipykernel_15732\4137921281.py:34: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata_sample = sc.AnnData(



Concatenating into master AnnData object...
Concatenation complete!


Cell 4: Save Raw Master Dataset

In [4]:
print("--- MASTER ANNDATA SUMMARY ---")
print(adata_master)
print("\nCell counts per tissue stage:")
print(adata_master.obs["tissue_type"].value_counts())

output_path = os.path.join(PROCESSED_DIR, "adata_raw_master.h5ad")
print(f"\nSaving master dataset to {output_path}...")
adata_master.write_h5ad(output_path, compression="gzip")
print("Phase 1 Complete!")

--- MASTER ANNDATA SUMMARY ---
AnnData object with n_obs × n_vars = 197954 × 33694
    obs: 'sample_id', 'gsm_id', 'patient_id', 'tissue_type'
    layers: None (.X)

Cell counts per tissue stage:
tissue_type
T          66015
P          50750
RightLN    49605
LeftLN     21337
SC         10247
Name: count, dtype: int64

Saving master dataset to ./processed_data\adata_raw_master.h5ad...
Phase 1 Complete!


Cell 5 : Update of Master file

In [5]:
# Standardize tissue_type categories
clean_tissue_map = {
    'P': 'Paratumor',
    'T': 'Primary Tumor',
    'RightLN': 'LN Metastasis',
    'LeftLN': 'LN Metastasis',
    'SC': 'RAI-Refractory Metastasis'
}

adata_master.obs['tissue_type'] = adata_master.obs['tissue_type'].map(clean_tissue_map)

print("--- UPDATED TISSUE TYPE COUNTS ---")
print(adata_master.obs['tissue_type'].value_counts())

# Resave master file
output_path = os.path.join(PROCESSED_DIR, "adata_raw_master.h5ad")
adata_master.write_h5ad(output_path, compression="gzip")
print("Phase 1 Metadata Cleaning Complete!")

--- UPDATED TISSUE TYPE COUNTS ---
tissue_type
LN Metastasis                70942
Primary Tumor                66015
Paratumor                    50750
RAI-Refractory Metastasis    10247
Name: count, dtype: int64
Phase 1 Metadata Cleaning Complete!
